<a href="https://colab.research.google.com/github/HiveCase/DLP-DA5013/blob/main/Week5/DLP_Week5_lecture_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install transformers datasets torch

In [3]:
from datasets import load_dataset

In [4]:
ds = load_dataset("facebook/voxpopuli","en", split="train",streaming=True)
from pprint import pprint
pprint(ds)

README.md:   0%|          | 0.00/31.9k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

IterableDataset({
    features: ['audio_id', 'language', 'audio', 'raw_text', 'normalized_text', 'gender', 'speaker_id', 'is_gold_transcript', 'accent'],
    num_shards: 30
})


In [5]:
subset = []
for i, example in enumerate(ds):
  if i>=50:
    break
  subset.append(example)

In [6]:
ds_es = load_dataset("facebook/voxpopuli","es", split="train",streaming=True)
pprint(ds_es)

IterableDataset({
    features: ['audio_id', 'language', 'audio', 'raw_text', 'normalized_text', 'gender', 'speaker_id', 'is_gold_transcript', 'accent'],
    num_shards: 10
})


In [7]:
for i, example in enumerate(ds_es):
  if i>=50:
    break
  subset.append(example)

In [8]:
ds_ro = load_dataset("facebook/voxpopuli","ro", split="train",streaming=True)
pprint(ds_ro)

IterableDataset({
    features: ['audio_id', 'language', 'audio', 'raw_text', 'normalized_text', 'gender', 'speaker_id', 'is_gold_transcript', 'accent'],
    num_shards: 5
})


In [9]:
for i, example in enumerate(ds_ro):
  if i>=50:
    break
  subset.append(example)

In [10]:
len(subset)

150

In [11]:
subset[0]

{'audio_id': '20180418-0900-PLENARY-3-en_20180418-08:50:36_17',
 'language': 0,
 'audio': <datasets.features._torchcodec.AudioDecoder at 0x7dfe7a51a120>,
 'raw_text': 'If you do not address this problem, the ground is there for populist nationalist forces to go on growing all over Europe.',
 'normalized_text': 'if you do not address this problem the ground is there for populist nationalist forces to go on growing all over europe.',
 'gender': 'female',
 'speaker_id': '124737',
 'is_gold_transcript': True,
 'accent': 'None'}

In [12]:
subset[60]

{'audio_id': '20120118-0900-PLENARY-19-es_20120118-18:39:15_6',
 'language': 3,
 'audio': <datasets.features._torchcodec.AudioDecoder at 0x7dfd7999def0>,
 'raw_text': 'Estas cumbres, señor Presidente, Señorías, necesitarían mayor coherencia entre los Estados, más trabajo diplomático y menos exhibición de festival.',
 'normalized_text': 'estas cumbres señor presidente señorías necesitarían mayor coherencia entre los estados más trabajo diplomático y menos exhibición de festival.',
 'gender': 'male',
 'speaker_id': '97170',
 'is_gold_transcript': True,
 'accent': 'None'}

In [13]:
subset[111]

{'audio_id': '20200210-0900-PLENARY-ro_20200210-18:27:29_3',
 'language': 6,
 'audio': <datasets.features._torchcodec.AudioDecoder at 0x7dfd799d55c0>,
 'raw_text': 'Așa cum s a spus aici, Europa este clar cu două viteze.',
 'normalized_text': 'așa cum s a spus aici europa este clar cu două viteze.',
 'gender': 'female',
 'speaker_id': '124785',
 'is_gold_transcript': True,
 'accent': 'None'}

In [14]:
import random
random.shuffle(subset)

In [15]:
import torch
from transformers import Wav2Vec2Processor, Wav2Vec2Model
from tqdm import tqdm

In [23]:
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h")
cell_count=0

preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.bias      | UNEXPECTED | 
lm_head.weight    | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [24]:
def extract_wav2vec_features(batch):
  global cell_count
  cell_count += 1
  print(cell_count)
  inputs = processor(batch["audio"]["array"], sampling_rate=16000, return_tensors="pt", padding=True)
  with torch.no_grad():
    outputs = model(**inputs)
  return {"features": outputs.last_hidden_state.squeeze().mean(dim=0).numpy(), "label": batch["language"]}

In [25]:
subset_with_features = [extract_wav2vec_features(example) for example in subset]

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150


In [28]:
from sklearn.model_selection import train_test_split

In [29]:
train_data, test_data = train_test_split(subset_with_features, test_size=0.2)
train_data, val_data= train_test_split(train_data, test_size=0.1)

In [30]:
def prepare_data(data):
  features = [item["features"] for item in data]
  labels = [item["label"] for item in data]
  return torch.tensor(features), torch.tensor(labels)

In [31]:
train_features, train_labels = prepare_data(train_data)
val_features, val_labels = prepare_data(val_data)
test_features, test_labels = prepare_data(test_data)

/tmp/ipykernel_1311/3983969988.py:4: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  return torch.tensor(features), torch.tensor(labels)


In [32]:
# building a classifier using CNN
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

class CNNModel(nn.Module):
  def __init__(self, input_dim, num_classes):
    super(CNNModel, self).__init__()
    self.conv1 = nn.Conv1d(in_channels=input_dim, out_channels=128, kernel_size=3, padding=1)
    self.conv2 = nn.Conv1d(in_channels=128, out_channels=256, kernel_size=3, padding=1)
    self.fc1 = nn.Linear(256*(input_length//4), 128)
    self.fc2 = nn.Linear(128, num_classes)

  def forward(self,x):
    x = F.relu(self.conv1(x))
    x = F.max_pool1d(x, kernel_size=2)
    x = F.relu(self.conv2(x))
    x = F.max_pool1d(x, kernel_size=2)
    x = x.view(x.size(0), -1)
    x = F.relu(self.fc1(x))
    x = self.fc2(x)
    return x

In [34]:
input_length = train_features.shape[1]

In [35]:
label_mapping = {6:0, 0:1, 3:2}

In [36]:
train_labels = train_labels.apply_(lambda x: label_mapping[x])
val_labels = val_labels.apply_(lambda x: label_mapping[x])
test_labels = test_labels.apply_(lambda x: label_mapping[x])

In [37]:
unique_train_labels = torch.unique(train_labels)
unique_test_labels = torch.unique(test_labels)
unique_val_labels = torch.unique(val_labels)
print(f"Unique Train labels: {unique_train_labels}")
print(f"Unique Val labels: {unique_val_labels}")
print(f"Unique Test labels: {unique_test_labels}")

Unique Train labels: tensor([0, 1, 2])
Unique Val labels: tensor([0, 1, 2])
Unique Test labels: tensor([0, 1, 2])


In [38]:
model = CNNModel(input_dim=1, num_classes=3)

In [39]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [40]:
num_epochs = 20
for epoch in range(num_epochs):
  model.train()
  optimizer.zero_grad()
  outputs = model(train_features.unsqueeze(1).float())
  loss = criterion(outputs, train_labels)
  loss.backward()
  optimizer.step()

  # validation
  model.eval()
  with torch.no_grad():
    val_outputs = model(val_features.unsqueeze(1).float())
    val_loss = criterion(val_outputs, val_labels)
  print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss.item()}, Val Loss: {val_loss.item()}")

Epoch 1/20, Loss: 1.0974453687667847, Val Loss: 2.223567008972168
Epoch 2/20, Loss: 2.050793170928955, Val Loss: 1.2271939516067505
Epoch 3/20, Loss: 1.188742995262146, Val Loss: 1.1204687356948853
Epoch 4/20, Loss: 1.1171373128890991, Val Loss: 1.0980654954910278
Epoch 5/20, Loss: 1.0980818271636963, Val Loss: 1.0947505235671997
Epoch 6/20, Loss: 1.0936684608459473, Val Loss: 1.0939351320266724
Epoch 7/20, Loss: 1.0904139280319214, Val Loss: 1.0931545495986938
Epoch 8/20, Loss: 1.0870248079299927, Val Loss: 1.091153621673584
Epoch 9/20, Loss: 1.0839396715164185, Val Loss: 1.085920810699463
Epoch 10/20, Loss: 1.0771812200546265, Val Loss: 1.0768918991088867
Epoch 11/20, Loss: 1.0675286054611206, Val Loss: 1.0655955076217651
Epoch 12/20, Loss: 1.056104063987732, Val Loss: 1.051941156387329
Epoch 13/20, Loss: 1.0410289764404297, Val Loss: 1.0313750505447388
Epoch 14/20, Loss: 1.0207089185714722, Val Loss: 1.0052660703659058
Epoch 15/20, Loss: 0.994973361492157, Val Loss: 0.97161680459976

In [41]:
from sklearn.metrics import accuracy_score

In [42]:
model.eval()
with torch.no_grad():
  test_outputs = model(test_features.unsqueeze(1).float())
  _, predicted = torch.max(test_outputs, 1)
  accuracy = accuracy_score(predicted.numpy(), test_labels.numpy())
print(f"Test Accuracy: {accuracy*100:.3f}%")

Test Accuracy: 53.333%
